# Bug Finding Assignment: RNN&LSTM

Instructions:
- This file provides 4 train scripts, and each of them has a bug
- Your task is to identify and fix all 4 bugs
- The bugs are related to:
  * RNN/LSTM torch API usage
  * Training pipeline issues
  * Learning rate scheduler usage

In [1]:
# Buggy code 1

import torch
import torch.nn as nn

class BuggyRNN(nn.Module):
    def __init__(self, input_size=10, hidden_size=20, num_classes=15):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=1,
        )
        self.activation = nn.Sigmoid()
        self.proj = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x shape: [batch_size, seq_len, input_size]
        out, hidden = self.rnn(x)
        act_out = self.activation(out)
        logits = self.proj(act_out)
        return logits

In [2]:
# Buggy code 2
import torch
import torch.nn as nn

class BuggyLSTM(nn.Module):
    def __init__(self, input_size=10, hidden_size=20, num_classes=15):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
        )
        self.activation = nn.Sigmoid()
        self.proj = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, hidden = self.lstm(x)
        act_out = self.activation(out)
        logits = self.proj(act_out)
        print(logits.shape)
        print(hidden.shape)
        return logits


In [3]:
# Buggy code 3
import torch
import torch.nn as nn

def compute_loss_buggy(model, batch, labels):
    # batch size: [batch_size, seq_len, input_size]
    # labels: [batch_size, seq_len]
    out, hidden = model(batch)
    logits = out.permute(0, 1, 2)
    loss = nn.CrossEntropyLoss()
    loss_fn = loss(logits, labels)
    return loss_fn

In [4]:
# Buggy code 4
import torch
import torch.nn as nn

def train_buggy_scheduler(model, dataset, num_epochs=3):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.1)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(num_epochs):
        for batch, labels in dataset:
            optimizer.zero_grad()
            out, hidden = model(batch)
            logits = out.permute(0, 2, 1)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
        scheduler.step()  
    
    return model